> **Historical research track.** This notebook documents the earlier trajectory-anomaly experiment. It is preserved as reproducible evidence and does not define the current SADAR Analyst Console product.

# Phase 6 (companion) — SADAR cross-project comparison

> Issue #27 · context: a teammate's parallel project **SADAR** (`huggingface.co/spaces/devrup404/sadar`)
> — same OpenSky-LEMD data, same LSTM/VAE-LSTM-AE approach. Detail + the PRE-REGISTERED Phase-7
> head-to-head live in `docs/research/trajectory-anomaly/lifecycle/07-eval-prep.md` "Layer 6".

**Question:** our Phase-6 model scores val AUROC **~0.66** on our §6 injections; SADAR reports
**0.792** synthetic. Is his model *better*, or are his injections *easier*?

**Method (firewall-clean):** score OUR trained model on (a) OUR injections and (b) HIS-style
injections (his `eval.yaml` params reproduced on our val segments). Compare to his published
numbers. This is **val-only synthetic** — it touches no sealed cohort. The real-anomaly
head-to-head (both models, blind) is **deferred to Phase 7** (07-eval-prep.md Layer 6); we do
**not** peek at the held-aside go-around cohort here.

Requires `research/trajectory-anomaly/notebooks/lifecycle/09_phase6_train.ipynb` to have run first (produces the cached frames + the
trained model under `.artifacts/research/trajectory-anomaly/phase6/`).

In [1]:
import sys
from pathlib import Path
import numpy as np, pandas as pd
from sklearn.metrics import roc_auc_score

REPO = Path.cwd()
while not (REPO / "backend/research/src/sadar_research/trajectory_anomaly/pipeline/preprocessing.py").exists() and REPO != REPO.parent:
    REPO = REPO.parent
sys.path.insert(0, str(REPO / "backend" / "research" / "src"))

from sadar_research.trajectory_anomaly.pipeline import split as sp
from sadar_research.trajectory_anomaly.pipeline.features import apply_segment_derivations
from sadar_research.trajectory_anomaly.pipeline.preprocessing import (AE_FEATURES, SCALER_FEATURES, MASKED_FEATURES,
                                         make_scaler, to_sequences, to_sequences_loss_mask)
from sadar_research.trajectory_anomaly.pipeline.inject import make_eval_set
from sadar_research.trajectory_anomaly.models import lstm_ae as ae

SEED = 42; MODELS = REPO / ".artifacts/research/trajectory-anomaly/phase6"; MPDLAT = 111_320.0
clean_df = pd.read_parquet(MODELS / "clean_df.parquet")
meta = pd.read_parquet(MODELS / "meta.parquet")
split = sp.split_by_monday(meta); sp.assert_firewall(split)
train_df = sp.subset(clean_df, split.train_ids); val_df = sp.subset(clean_df, split.val_ids)
T = int(np.percentile(train_df.groupby("segment_id").size(), 95))
scaler = make_scaler().fit(train_df[SCALER_FEATURES])
import torch
AGG = torch.load(str(MODELS/"lstm_ae_best.pt"), map_location="cpu", weights_only=False)["extra"]["agg"]
model = ae.load_checkpoint(str(MODELS / "lstm_ae_best.pt"))   # nb09 grid winner (small/mean post-fix)
print(f"loaded our model (T={T}, agg={AGG}); val segments: {len(split.val_ids)}")

loaded our model (T=260, agg=mean); val segments: 5942


## His published numbers (reference, from his `reports/model_comparison.json`)

His real-anomaly AUROC = normal-2020 vs his real emergency+go-around cohort (~4 emergency +
~100 go-around flights, his report). Note it's **go-around-dominated** and PR-AUC is the honest
lens at ~12% prevalence.

In [2]:
SADAR_REPORTED = pd.DataFrame([
    {"model":"Baseline","real_roc":0.515,"real_pr":0.133,"synth_mean_roc":0.593},
    {"model":"LSTM","real_roc":0.648,"real_pr":0.260,"synth_mean_roc":0.779},
    {"model":"VAE-LSTM (selected)","real_roc":0.659,"real_pr":0.299,"synth_mean_roc":0.792},
]).set_index("model")
SADAR_REPORTED

,real_roc,real_pr,synth_mean_roc
model,,,
Baseline,0.515,0.133,0.593
LSTM,0.648,0.260,0.779
VAE-LSTM (selected),0.659,0.299,0.792


## Our model on OUR injections (reproduce nb09 headline) and HIS-style injections

His `eval.yaml` injection grid is far more aggressive than our §6: route deviation **20-80 km**
(ours 1-3 km), symmetric altitude 300/800/1500 m, speed ×1.6/2.2/0.4, holding, freeze. We
reproduce his params on our raw-space val segments via the same scaffold.

In [3]:
# our injections (winner agg, from nb09)
our_set = make_eval_set(clean_df, split.val_ids, scaler, T, seed=SEED, fold="val", inject_rate=0.5)
auroc_our_our = roc_auc_score(our_set.y, ae.reconstruction_error(model, our_set.X, our_set.loss_mask, agg=AGG))
print(f"our model / OUR injections : {auroc_our_our:.4f}  (matches nb09 bake-off)")

our model / OUR injections : 0.6642  (matches nb09 bake-off)


In [4]:
# --- HIS-style perturbations (his eval.yaml params), raw space, measured cols only ---
def _ramp(n,o):
    r=np.zeros(n);
    if o>=n-1: r[n-1:]=1.0
    else: r[o:]=np.linspace(0,1,n-o)
    return r
def route(seg,o,mag,rng):
    n=len(seg); rr=_ramp(n,o); b=rng.uniform(0,2*np.pi); lat=seg["lat"].to_numpy()
    seg["lat"]=lat+np.cos(b)*mag*rr/MPDLAT
    seg["lon"]=seg["lon"].to_numpy()+np.sin(b)*mag*rr/(MPDLAT*np.cos(np.radians(lat)))
def altitude(seg,o,mag,rng):
    n=len(seg); off=rng.choice([-1.,1.])*mag*_ramp(n,o)
    seg["baroaltitude"]=seg["baroaltitude"].to_numpy()+off
    d=np.zeros(n); d[1:]=np.diff(off)/10.0; seg["vertrate"]=seg["vertrate"].to_numpy()+d
def speed(seg,o,f,rng): seg.loc[seg.index[o:],"velocity"]=np.clip(seg["velocity"].to_numpy()[o:]*f,0,None)
def holding(seg,o,period,rng):
    n=len(seg); idx=seg.index[o:]; k=len(idx); spd=float(seg["velocity"].iloc[o])
    h0=np.radians(float(seg["heading"].iloc[o])); om=2*np.pi/period; el=np.arange(k)*10.0; h=h0+om*el
    lat0=float(seg["lat"].iloc[o]); lon0=float(seg["lon"].iloc[o])
    e=np.concatenate([[0],np.cumsum((spd*10*np.sin(h))[:-1])]); nn=np.concatenate([[0],np.cumsum((spd*10*np.cos(h))[:-1])])
    seg.loc[idx,"lat"]=lat0+nn/MPDLAT; seg.loc[idx,"lon"]=lon0+e/(MPDLAT*np.cos(np.radians(lat0)))
    seg.loc[idx,"heading"]=np.degrees(h)%360; seg.loc[idx,"vertrate"]=0.0
def freeze(seg,o,_,rng):
    cols=["lat","lon","baroaltitude","velocity","vertrate","heading"]
    seg.loc[seg.index[o:],cols]=seg.loc[seg.index[o],cols].to_numpy()

TYPES = ([("route",m,route) for m in (20000,40000,80000)]
       + [("altitude",m,altitude) for m in (300,800,1500)]
       + [("speed",f,speed) for f in (1.6,2.2,0.4)]
       + [("holding",p,holding) for p in (240,120)] + [("freeze",0,freeze)])

rng=np.random.default_rng(SEED); ids=list(split.val_ids); rng.shuffle(ids)
by_seg={s:g for s,g in val_df.groupby("segment_id",sort=False)}
pool=[s for s in ids if s in by_seg]
norm=pd.concat([by_seg[s].sort_values("time").reset_index(drop=True) for s in pool],ignore_index=True)
Xn,_,_=to_sequences(norm,T,scaler); mn=to_sequences_loss_mask(norm,T)
score_n=ae.reconstruction_error(model,Xn,mn,agg=AGG)

per_type={}; pos_all=[]
for ti,(name,param,fn) in enumerate(TYPES):
    sub=pool[ti*200:(ti+1)*200] or pool[:200]; frames=[]
    for s in sub:
        seg=by_seg[s].sort_values("time").reset_index(drop=True).copy()
        o=int(np.clip(round(len(seg)*0.5),0,len(seg)-1)); fn(seg,o,param,rng); seg=apply_segment_derivations(seg)
        mc=[f+"_missing" for f in MASKED_FEATURES if f+"_missing" in seg.columns]; seg.loc[seg.index[o:],mc]=False
        frames.append(seg)
    pooldf=pd.concat(frames,ignore_index=True); Xp,_,_=to_sequences(pooldf,T,scaler); mp=to_sequences_loss_mask(pooldf,T)
    sp_=ae.reconstruction_error(model,Xp,mp,agg=AGG)
    y=np.r_[np.zeros(len(score_n)),np.ones(len(sp_))]; per_type[f"{name} {param}"]=roc_auc_score(y,np.r_[score_n,sp_]); pos_all.append(sp_)

allpos=np.concatenate(pos_all)
auroc_our_his=roc_auc_score(np.r_[np.zeros(len(score_n)),np.ones(len(allpos))], np.r_[score_n,allpos])
pd.Series(per_type, name="our model on his-style injections (AUROC)").round(4)

route 20000      0.5753
route 40000      0.6239
route 80000      0.7833
altitude 300     0.5849
altitude 800     0.5890
altitude 1500    0.5776
speed 1.6        0.8886
speed 2.2        0.9688
speed 0.4        0.8709
holding 240      0.9402
holding 120      0.9403
freeze 0         0.8602
Name: our model on his-style injections (AUROC), dtype: float64

In [5]:
print(f"our model / HIS-style injections : mean {np.mean(list(per_type.values())):.4f} | overall {auroc_our_his:.4f}")

our model / HIS-style injections : mean 0.7669 | overall 0.7669


## The cross-tab + conclusion

In [6]:
xtab = pd.DataFrame(
    {"our injections (hard, §6)":  [round(auroc_our_our,3), "—"],
     "his injections (easy, his cfg)": [round(auroc_our_his,3), 0.792]},
    index=["our model (small/mean)", "his VAE-LSTM (reported)"])
print(xtab.to_string())
print(f"""
Conclusion: fed HIS injections, our model jumps {auroc_our_our:.3f} -> {auroc_our_his:.3f},
landing next to his 0.792. The 0.684-vs-0.792 headline gap is BENCHMARK DIFFICULTY, not model
quality (his route deviations are 20-80 km vs our 1-3 km; speed/holding/freeze are near-trivial;
on realistic altitude-300 m BOTH sit ~0.5-0.6). The un-gameable comparison is real-anomaly
AUROC (his VAE-LSTM 0.659 / PR 0.299, go-around-dominated) -> PRE-REGISTERED, blind, both
models, Phase 7 (07-eval-prep.md Layer 6). Nothing sealed was touched here.""")

                        our injections (hard, §6)  his injections (easy, his cfg)
our model (small/mean)                      0.664                           0.767
his VAE-LSTM (reported)                         —                           0.792

Conclusion: fed HIS injections, our model jumps 0.664 -> 0.767,
landing next to his 0.792. The 0.684-vs-0.792 headline gap is BENCHMARK DIFFICULTY, not model
quality (his route deviations are 20-80 km vs our 1-3 km; speed/holding/freeze are near-trivial;
on realistic altitude-300 m BOTH sit ~0.5-0.6). The un-gameable comparison is real-anomaly
AUROC (his VAE-LSTM 0.659 / PR 0.299, go-around-dominated) -> PRE-REGISTERED, blind, both
models, Phase 7 (07-eval-prep.md Layer 6). Nothing sealed was touched here.
